[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C02/E02-2_Criticite_Fissures.ipynb)

# Criticité d'une fissure : valider sans fuite

## Le contexte

Lors d'une inspection, des fissures sont mesurées sur des pièces en service : dimensions, état de surface, forme du contour, intensité de contrainte locale. Pour chacune, il faut décider s'il faut intervenir sans attendre (fissure critique) ou si l'on peut surveiller (fissure stable). Une fissure critique classée stable est un risque ; une fissure stable classée critique est un arrêt inutile.

Un modèle peut aider à trier, à condition de savoir honnêtement ce qu'il vaut avant de s'en servir. Avec 569 fissures seulement, on n'a pas de quoi gaspiller des données : la validation croisée permet de comparer des modèles sans entamer la base de test.

C'est un problème de classification binaire : la grandeur à prédire est une catégorie.

Défi : prédire `etat_fissure` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ n'utilise qu'une seule variable, la longueur de la fissure, et atteint un F-score macro de 0,856 : à vous d'ajouter les autres. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Criticite_Fissures.xlsx`, dans le dossier `donnees` du dépôt : 569 fissures, 13 colonnes. Une ligne est une fissure.

| Colonne | Signification | Unité |
|---|---|---|
| `longueur_fissure_mm` | longueur de la fissure | mm |
| `profondeur_fissure_mm` | profondeur de la fissure | mm |
| `surface_proj_mm2` | surface projetée | mm² |
| `rugosite_moy_ra_um` | rugosité moyenne des lèvres | µm |
| `compacite_morpho` | compacité de la forme | sans unité |
| `symetrie_contour` | symétrie du contour | sans unité |
| `intensite_contrainte_mpa_sqrtm` | intensité de contrainte | MPa·√m |
| `facteur_intensite_locale` | indicateur d'intensité locale, construit | sans unité |
| `indice_contrainte_equivalente` | indicateur de contrainte équivalente, construit | sans unité |
| `rapport_longueur_profondeur` | longueur divisée par profondeur | sans unité |
| `indice_morphologique_global` | indicateur de forme, construit | sans unité |
| `classe_criticite_fissure` | classe en trois niveaux | catégorie |
| `etat_fissure` | cible : `Critique` ou `Stable` | catégorie |

Origine : jeu public transformé. Ce support pédagogique réinterprète les descripteurs d'origine en grandeurs de fissure ; il enseigne une méthode, et aucune conclusion de mécanique de la rupture n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la répartition des deux états avant toute modélisation. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Criticite_Fissures.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Criticite_Fissures.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "etat_fissure"
print("dimensions :", df.shape)
print("cases vides au total :", int(df.isna().sum().sum()))
print(df[cible].value_counts().to_string())
print(df["classe_criticite_fissure"].value_counts().to_string())
df.head(3)

Lecture du résultat : 569 fissures, 13 colonnes, aucune case vide. 212 fissures sont critiques et 357 stables : les classes sont inégales, sans être rares. La colonne `classe_criticite_fissure` range 538 fissures sur 569 dans une seule classe : elle n'est pas utilisée ici, les onze mesures numériques suffisent.

### Étape 2 : séparer l'apprentissage et le test

Un quart des fissures est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec l'état à prédire. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Le tirage est stratifié : les deux états gardent la même proportion dans les deux bases.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[cible, "classe_criticite_fissure"])
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "fissures, dont", int((y_train == "Critique").sum()), "critiques")
print("test :", len(X_test), "fissures, dont", int((y_test == "Critique").sum()), "critiques")

### Étape 3 : choisir la variable de départ

On code l'état en 0 (stable) et 1 (critique), puis on calcule la corrélation de chaque mesure avec lui. Ce calcul croise l'état à prédire : il se lit sur l'apprentissage seul, sans la base de test. Le modèle de départ n'en retient qu'une : la longueur de la fissure, première grandeur relevée à toute inspection. Elle n'arrive qu'en deuxième position, derrière une grandeur construite : ce choix laisse de la marge à qui ajoute les autres. On trace sa répartition dans les deux états.

In [ ]:
critique = (y_train == "Critique").astype(int)
correlations = X_train.corrwith(critique).sort_values()
print(correlations.round(3).to_string())

# Variable du modèle de départ : la longueur de la fissure.
variable = "longueur_fissure_mm"

fig, ax = plt.subplots(figsize=(6.5, 4))
bornes = np.linspace(X_train[variable].min(), X_train[variable].max(), 30)
ax.hist(X_train.loc[critique == 1, variable], bins=bornes, color="0.15", alpha=0.75, label="critique")
ax.hist(X_train.loc[critique == 0, variable], bins=bornes, color="0.7", alpha=0.75, hatch="//", label="stable")
ax.set_xlabel(variable)
ax.set_ylabel("nombre de fissures")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()

Lecture du résultat : sur les 426 fissures d'apprentissage, la longueur est liée à l'état (corrélation de 0,733), derrière le facteur d'intensité locale (0,762). Les deux histogrammes se recouvrent en partie : dans ce jeu, les fissures critiques sont plus longues que les stables, et une zone d'incertitude demeure entre les deux. Ce sens est propre au jeu, construit pour enseigner la méthode : il ne dit rien de la physique de la rupture. Une seule mesure trie déjà correctement, sans trier parfaitement.

### Étape 4 : construire le modèle de départ

Le modèle de départ est volontairement simple : une régression logistique qui estime la probabilité qu'une fissure soit critique à partir de la seule variable choisie à l'étape 3. Les mesures n'ont pas la même échelle ; la mise à l'échelle et le modèle forment un seul pipeline, qui apprend sur `X_train` seul. Les autres mesures sont laissées de côté : elles sont votre marge de progression.

In [ ]:
# Votre code :

### Étape 5 : valider par validation croisée

Avant de toucher au test, on estime la qualité du modèle sur l'apprentissage seul. La validation croisée coupe l'apprentissage en cinq plis ; chaque pli sert une fois de validation à un modèle ajusté sur les quatre autres. On obtient cinq scores, donc une moyenne et une dispersion. Le pipeline est ré-ajusté dans chaque pli : la mise à l'échelle ne voit jamais les fissures de validation.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

plis = StratifiedKFold(n_splits=5, shuffle=True, random_state=GRAINE)
notes = cross_val_score(modele, X_train[[variable]], y_train, cv=plis, scoring="f1_macro")
print("scores des 5 plis :", notes.round(3))
print(f"moyenne : {notes.mean():.3f}, écart-type : {notes.std():.3f}")

Lecture du résultat : les cinq scores vont de 0,840 à 0,896, pour une moyenne de 0,873. L'écart-type de 0,021 dit de combien le score bouge d'un découpage à l'autre : deux modèles qui diffèrent de moins que cela ne se distinguent pas. Cette estimation n'a pas utilisé le test.

### Étape 6 : mesurer sur la base de test

Le modèle prédit les fissures qu'il n'a jamais vues. On lit le F-score macro, puis la matrice de confusion. Ici, l'erreur la plus coûteuse est de classer stable une fissure critique : on lit donc aussi le rappel de la classe `Critique`, part des fissures critiques retrouvées.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

y_pred = modele.predict(X_test[[variable]])
score_depart = f1_score(y_test, y_pred, average="macro")
rappel_depart = recall_score(y_test, y_pred, pos_label="Critique")
print(f"F-score macro : {score_depart:.3f}")
print(f"rappel de la classe Critique : {rappel_depart:.3f}")
print(classification_report(y_test, y_pred, digits=3))
matrice = confusion_matrix(y_test, y_pred, labels=["Critique", "Stable"])
print("matrice de confusion (lignes : réel, colonnes : prédit ; Critique, Stable)")
print(matrice)

Lecture du résultat : le F-score macro est de 0,856 sur le test, contre 0,873 en validation croisée. L'écart est de l'ordre de la dispersion d'un pli à l'autre : avec 143 fissures de test, le score n'est connu qu'à quelques centièmes près. Le modèle retrouve 42 fissures critiques sur 53 (rappel de 0,792) et en laisse passer 11. C'est ce chiffre, mesuré sur des fissures que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : la fuite à voir de ses yeux, puis un modèle complet

Le modèle de départ de l'étape 4 reste le score de référence du défi. La première cellule montre pourquoi la validation croisée doit refaire chaque choix dans chaque pli : on cherche à prédire l'état à partir de 300 colonnes de pur bruit, où aucun modèle ne peut réussir. Si l'on choisit d'abord les 10 meilleures colonnes sur tout l'apprentissage, puis qu'on valide, le score est flatteur à tort. Si le choix est refait dans chaque pli, il retombe au niveau du hasard. La seconde cellule compare ensuite des modèles complets par validation croisée, avant de mesurer le meilleur une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, modele.predict(X_test[[variable]]), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Un modèle apprend sur 426 fissures et se juge sur 143 fissures qu'il n'a jamais vues, tirées avec une graine fixée et stratifiées.
2. La validation croisée estime la qualité sur l'apprentissage seul : elle donne une moyenne et une dispersion, et elle garde le test intact.
3. Tout ce qui apprend quelque chose des données, mise à l'échelle ou choix de colonnes, doit être refait dans chaque pli : on le place dans le pipeline.
4. Deux modèles qui diffèrent de moins que l'écart-type des plis ne se distinguent pas.
5. La base de test sert une seule fois, au modèle retenu : choisir en regardant le test, c'est l'utiliser pour apprendre.